In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])

print(df.shape)
df.head()

(5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["message"]
y = df["label"]

# Step A: 70% train, 30% temporary holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step B: split the holdout in half -> 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

vectorizer = TfidfVectorizer(stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("Training messages:", X_train_vec.shape[0])
print("Validation messages:", X_val_vec.shape[0])
print("Test messages:", X_test_vec.shape[0])

Training messages: 3900
Validation messages: 836
Test messages: 836


In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = MultinomialNB()
model.fit(X_train_vec, y_train)

val_pred = model.predict(X_val_vec)

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print()
print(classification_report(y_val, val_pred))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_val, val_pred, labels=["ham", "spam"]))


Validation accuracy: 0.9665071770334929

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       724
        spam       1.00      0.75      0.86       112

    accuracy                           0.97       836
   macro avg       0.98      0.88      0.92       836
weighted avg       0.97      0.97      0.96       836

Confusion matrix (rows = actual, columns = predicted):
[[724   0]
 [ 28  84]]


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

candidates = {
    "NaiveBayes alpha=1.0 (current)": MultinomialNB(alpha=1.0),
    "NaiveBayes alpha=0.5": MultinomialNB(alpha=0.5),
    "NaiveBayes alpha=0.1": MultinomialNB(alpha=0.1),
    "NaiveBayes alpha=0.05": MultinomialNB(alpha=0.05),
    "LogisticRegression balanced": LogisticRegression(class_weight="balanced", max_iter=1000),
    "LinearSVC balanced": LinearSVC(class_weight="balanced"),
}

for name, m in candidates.items():
    m.fit(X_train_vec, y_train)
    pred = m.predict(X_val_vec)
    acc = accuracy_score(y_val, pred)
    prec = precision_score(y_val, pred, pos_label="spam")
    rec = recall_score(y_val, pred, pos_label="spam")
    f1 = f1_score(y_val, pred, pos_label="spam")
    print(f"{name:32s} acc={acc:.3f}  spam_precision={prec:.3f}  spam_recall={rec:.3f}  spam_f1={f1:.3f}")

NaiveBayes alpha=1.0 (current)   acc=0.967  spam_precision=1.000  spam_recall=0.750  spam_f1=0.857
NaiveBayes alpha=0.5             acc=0.972  spam_precision=1.000  spam_recall=0.795  spam_f1=0.886
NaiveBayes alpha=0.1             acc=0.983  spam_precision=0.980  spam_recall=0.893  spam_f1=0.935
NaiveBayes alpha=0.05            acc=0.982  spam_precision=0.971  spam_recall=0.893  spam_f1=0.930
LogisticRegression balanced      acc=0.981  spam_precision=0.936  spam_recall=0.920  spam_f1=0.928
LinearSVC balanced               acc=0.983  spam_precision=0.971  spam_recall=0.902  spam_f1=0.935


In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

final_model = MultinomialNB(alpha=0.1)
final_model.fit(X_train_vec, y_train)

test_pred = final_model.predict(X_test_vec)

print("TEST accuracy:", accuracy_score(y_test, test_pred))
print()
print(classification_report(y_test, test_pred))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, test_pred, labels=["ham", "spam"]))

TEST accuracy: 0.9856459330143541

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       724
        spam       0.97      0.92      0.94       112

    accuracy                           0.99       836
   macro avg       0.98      0.96      0.97       836
weighted avg       0.99      0.99      0.99       836

Confusion matrix (rows = actual, columns = predicted):
[[721   3]
 [  9 103]]


In [13]:
def check_message(text):
    vec = vectorizer.transform([text])
    label = final_model.predict(vec)[0]
    probs = final_model.predict_proba(vec)[0]
    spam_chance = probs[list(final_model.classes_).index("spam")] * 100
    verdict = "SCAM / SPAM" if label == "spam" else "Looks normal"
    print(f"{verdict}  (spam chance: {spam_chance:.1f}%)")
    print("Message:", text)
    print("-" * 50)

tests = [
    "Congratulations! You have won a FREE prize of $1000. Click the link now to claim your reward!",
    "Your account has been suspended. Verify your details immediately to avoid closure.",
    "Hey, are we still meeting  at the library at 4pm? but bring your money",
    "Please help with lecture notes when you get a chance and also the atm card pin and forward the otp once send, thanks.",
]

for t in tests:
    check_message(t)


SCAM / SPAM  (spam chance: 100.0%)
Message: Congratulations! You have won a FREE prize of $1000. Click the link now to claim your reward!
--------------------------------------------------
SCAM / SPAM  (spam chance: 65.3%)
Message: Your account has been suspended. Verify your details immediately to avoid closure.
--------------------------------------------------
Looks normal  (spam chance: 0.0%)
Message: Hey, are we still meeting  at the library at 4pm? but bring your money
--------------------------------------------------
Looks normal  (spam chance: 5.5%)
Message: Please help with lecture notes when you get a chance and also the atm card pin and forward the otp once send, thanks.
--------------------------------------------------


In [14]:
import pandas as pd

url = "https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/texts.json"

try:
    texts = pd.read_json(url)
except ValueError:
    texts = pd.read_json(url, lines=True)

texts["label"] = texts["label"].map({1: "phishing", 0: "normal"})

print("Shape:", texts.shape)
print(texts["label"].value_counts())
print()

for lab in ["phishing", "normal"]:
    print("=== Examples:", lab, "===")
    for t in texts[texts["label"] == lab]["text"].sample(3, random_state=1):
        print("-", t[:200].replace("\n", " "))
    print()

Shape: (20137, 2)
label
normal      12465
phishing     7672
Name: count, dtype: int64

=== Examples: phishing ===
- perfect logo charset = koi 8 - r " > thinking of breathing new life into your business ? start from revamping its front - end - logo and visuai identity . loqodentity offers creative custom design of 
- i ' d like to tell you about a safe alternative to bextra rub o 24 pain relief lotionwhere it hurts , when it hurts us patent 6 , 444 , 238 bl clinically tested now available in the usa natural pain r
- small stocks can give you short term returns rocket stocks newsletter u r g e n t i n v e s t o r a l e r t vnbl stock expected to explode ! ! ! vinoble is the next hot play in homeland security symbo

=== Examples: normal ===
- On Sat, 24 Aug 2002, Adam L. Beberg wrote:--]And yet STILL noone is out there creating _public domain_ content. Is there --]even one person out there can can even begin to talk without being a --]comp
- URL: http://www.newsisfree.com/click/-2,842199

In [15]:
short = texts[texts["text"].str.len() < 300]

print("Short messages:", len(short))
print(short["label"].value_counts())
print()

print("=== 8 phishing examples (short) ===")
for t in short[short["label"] == "phishing"]["text"].sample(8, random_state=3):
    print("-", t.replace("\n", " "))

Short messages: 5353
label
normal      3158
phishing    2195
Name: count, dtype: int64

=== 8 phishing examples (short) ===
- Urgent! Please call 09066612661 from your landline, your complimentary 4* Lux Costa Del Sol holiday or £1000 CASH await collection. ppm 150 SAE T&Cs James 28, EH74RR
- start shopping at costco today with a complimentary gold membership . start shopping at costco today with a free gold membership . free gold membership or upgrade and extend your existing membership . ( ) ipmlgbit
- I don't know u and u don't know me. Send CHAT to 86688 now and let's find each other! Only 150p/Msg rcvd.
- 	Do you want a New Nokia 3510i colour phone DeliveredTomorrow? With 300 free minutes to any mobil+ Free Camcorder reply or call 08000930705.
- craig all the meds you need : http : / / www . salver . info / ? webmaster dominick
- 	goldviking (29/M) is inviting you to be his friend. Reply YES-762 or NO-762 See him: www.SMS.ac/u/goldviking STOP? Send STOP FRND to 62468
- free x - bo

In [16]:
import pandas as pd

scam_msgs = [
    "Dear customer, your account will be blocked today. Reply with your BVN and ATM PIN to keep it active.",
    "GTBank alert: unusual login detected. Confirm your card number and OTP at http://gtb-secure-verify.com to avoid suspension.",
    "Hello, I mistakenly sent money to your account. Please send back the code that was sent to your phone so I can reverse it.",
    "Congratulations! You have been selected for a N500,000 grant. Send your account number and BVN to receive it.",
    "Please read me the 6 digit number you just received, I need it to confirm something for you.",
    "Your MTN data promo is ready. Click here to claim 50GB free and enter your password.",
    "Urgent: your debit card has been deactivated. Call this number now and give the agent the 3 digits at the back of your card.",
    "Hi dear, I am stuck at the airport and need money urgently. Send me your card details and I will pay you back double.",
    "Invest N20,000 today and receive N200,000 in 24 hours. Send your bank details now to join.",
    "We are hiring remote workers. Pay a N5,000 registration fee and send your login password to start today.",
    "Your account has been suspended. Verify your identity immediately at the link below or it will be closed.",
    "Final notice: your NIN is about to expire. Send your NIN and BVN to this WhatsApp number to avoid being blocked.",
    "Brother please forward me the verification code that just came to your phone, my own is not working.",
    "Dear user, your Access Bank token has expired. Enter your PIN here to renew: http://bit.ly/renew-token",
    "You won the Airtel lucky draw! Send N2,000 to claim your new iPhone.",
    "Mummy it's me, I lost my phone. Please send N30,000 to this account urgently, don't call me.",
    "Your PayPal account is limited. Log in with your email and password at paypal-secure-help.com to restore access.",
    "Copy the numbers on the front and back of your card and send them to me so I can pay you faster.",
    "Bitcoin giveaway: send 0.1 BTC to this address and get 1 BTC back instantly.",
    "CBN is giving free money to citizens. Register with your BVN, PIN and phone number now.",
    "Dear student, your scholarship is approved. Send your school portal password to claim it.",
    "Your package is held at customs. Pay N3,500 delivery fee at this link to release it.",
    "Please send me your Opay PIN so I can help you unlock your account, it was flagged.",
    "Hi, tell me the code Google just sent you, I'm trying to verify that it's really you.",
    "Your electricity token purchase failed. Re-enter your card number, expiry date and CVV to complete it.",
    "Congrats winner!! Reply with your full name, address and card number to receive your free gift.",
    "Your WhatsApp will be deleted in 24 hours. Click this link and enter the code we send you to keep it.",
    "Make N100k weekly from home. Send me your bank login so I can set up your earnings.",
    "I am a prince with 5 million dollars to transfer, I need your account details and a small fee to start.",
    "Security alert: someone tried to use your card. Reply with the OTP you just received to cancel the transaction.",
]

normal_msgs = [
    "Hey, are we still meeting at the library at 4pm?",
    "Please send me the lecture notes when you get a chance, thanks.",
    "Your OTP is 482913. It expires in 5 minutes. Do not share this code with anyone.",
    "Debit alert: N5,000 has been withdrawn from your account at ATM. If this was not you, visit your nearest branch.",
    "I just checked my bank app, the transfer has arrived. Thanks so much!",
    "Don't forget to pin the meeting location on the group chat.",
    "Can you help me with the Python assignment? I'm stuck on loops.",
    "Mum, I've arrived safely. I'll call you tonight.",
    "Our class is moved to Thursday at 10am in Room 204.",
    "Please bring your student ID card to the exam hall tomorrow.",
    "Happy birthday! Wishing you all the best, enjoy your day.",
    "Credit alert: N10,000 received from Tunde. Balance is N24,350.",
    "I forgot my password for the school portal, how do I reset it?",
    "Let's meet at the mall at 6. Bring your money so we can split the bill.",
    "The weather forecast says heavy rain tomorrow, carry an umbrella.",
    "Abeg send me the assignment, I never see am.",
    "How far? You dey come church later?",
    "Your Uber is arriving in 3 minutes. Your driver is in a white Toyota.",
    "Reminder: your dentist appointment is tomorrow at 2pm.",
    "I changed my ATM card at the bank today, the new one will be ready next week.",
    "Thanks for the call. Send the project files to my email when you can.",
    "Your order has been shipped and will arrive on Friday.",
    "Your verification code is 731902. If you didn't request this, please ignore this message.",
    "Can you share the slides from today's tech meetup?",
    "We are hiring a junior developer. Send your CV to the email on our careers page, no payment is required.",
    "Did you see the match last night? That goal was crazy.",
    "Please remember to back up your files before updating your laptop.",
    "I'll send the money for the tickets tomorrow morning, is that fine?",
    "Dr. Bello wants everyone to submit the report by Monday.",
    "Your electricity token is 1234-5678-9012-3456-7890. Units: 25.4 kWh.",
]

custom = pd.DataFrame({
    "message": scam_msgs + normal_msgs,
    "label": ["spam"] * len(scam_msgs) + ["ham"] * len(normal_msgs),
})

print(custom.shape)
print(custom["label"].value_counts())

(60, 2)
label
spam    30
ham     30
Name: count, dtype: int64


In [20]:
import pandas as pd

new = pd.read_csv("Dataset_10191 (1).csv", encoding="latin-1")

print("Shape:", new.shape)
print("Columns:", new.columns.tolist())
print(new["LABEL"].value_counts())
print()

for lab in new["LABEL"].unique():
    print("===", lab, "===")
    for t in new[new["LABEL"] == lab]["TEXT"].sample(4, random_state=1):
        print("-", t)
    print()

Shape: (10191, 5)
Columns: ['LABEL', 'TEXT', 'URL', 'EMAIL', 'PHONE']
LABEL
ham         3397
smishing    3397
spam        3397
Name: count, dtype: int64

=== ham ===
- Now only i reached home. . . I am very tired now. .will talk to u tomorrow
- Tee hee. Off to lecture, cheery bye bye.
- I ain't answerin no phone at what is actually a pretty reasonable hour but I'm sleepy
- Lol now I'm after that hot air balloon!

=== smishing ===
- you let won! as a treasure vodafone customer our computer possess cull you to succeed a Â£550 prize. to gather just call 09061743389
- urgent message for cibc account holder, kindly login to substantiate your login information login to our secure server to discipline your information http://cibc.mobileupdate.granpianostudio.com/cibcfullz/internet support
- dear customer, due to bvn system upgrade, your atm card ingest embody de-activated by cbn. to re-activate call customer care 08167340838 for help.
- do you want a new video handset? 750 anytime any network

In [19]:

from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving Dataset_10191 (1).csv to Dataset_10191 (1).csv
['Dataset_10191 (1).csv']


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Load the newly uploaded CSV
df_new = pd.read_csv("Dataset_10191 (1).csv")

# Clean any missing text values
df_new["TEXT"] = df_new["TEXT"].fillna("")

# Inspect the structure to see how URL, EMAIL, PHONE are formatted (e.g., boolean, 0/1, or strings)
print(df_new.head(3))

  LABEL                                               TEXT URL EMAIL PHONE
0   ham  Hi! You just spoke to DEEPAK. We'd like to kno...  No    No    No
1   ham                  Yay can't wait to party together!  No    No    No
2   ham                       At what time are you coming.  No    No    No


In [22]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Convert "Yes"/"No" metadata columns to 1/0
for col in ["URL", "EMAIL", "PHONE"]:
    df_new[col] = df_new[col].apply(
        lambda x: 0 if str(x).strip().lower() in ["no", "false", "0", "nan", "none"] else 1
    )

X_text = df_new["TEXT"]
X_meta = df_new[["URL", "EMAIL", "PHONE"]].astype(float)
y = df_new["LABEL"]

# 2. Split 75% Train, 25% Validation (7.5 / 2.5 ratio)
X_train_t, X_val_t, X_train_m, X_val_m, y_train, y_val = train_test_split(
    X_text, X_meta, y, test_size=0.25, random_state=42, stratify=y
)

# 3. Vectorize text (1-2 word n-grams to catch phrases like "atm pin", "send otp", "verify account")
vectorizer_new = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=12000)

X_train_vec = hstack([vectorizer_new.fit_transform(X_train_t), X_train_m.values])
X_val_vec = hstack([vectorizer_new.transform(X_val_t), X_val_m.values])

print(f"Training samples:   {X_train_vec.shape[0]} (75%)")
print(f"Validation samples: {X_val_vec.shape[0]} (25%)")

Training samples:   7643 (75%)
Validation samples: 2548 (25%)


In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

models = {
    "MultinomialNB (alpha=0.1)": MultinomialNB(alpha=0.1),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "LinearSVC (Calibrated)": CalibratedClassifierCV(LinearSVC(random_state=42))
}

for name, m in models.items():
    m.fit(X_train_vec, y_train)
    val_preds = m.predict(X_val_vec)
    acc = accuracy_score(y_val, val_preds)
    print(f"{name:28s} | Validation Accuracy: {acc:.4f}")

# Select LogisticRegression (or LinearSVC) as our primary model
best_model = models["LogisticRegression"]
print("\n--- Detailed Validation Report (Logistic Regression) ---")
print(classification_report(y_val, best_model.predict(X_val_vec)))


MultinomialNB (alpha=0.1)    | Validation Accuracy: 0.9647
LogisticRegression           | Validation Accuracy: 0.9690
LinearSVC (Calibrated)       | Validation Accuracy: 0.9788

--- Detailed Validation Report (Logistic Regression) ---
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       849
    smishing       0.97      0.95      0.96       849
        spam       0.95      0.96      0.96       850

    accuracy                           0.97      2548
   macro avg       0.97      0.97      0.97      2548
weighted avg       0.97      0.97      0.97      2548



In [24]:
from sklearn.metrics import confusion_matrix

def extract_features(messages):
    # TF-IDF features
    text_vec = vectorizer_new.transform(messages)

    # Metadata flags: URL, EMAIL, PHONE
    meta_rows = []
    for msg in messages:
        has_url = 1 if re.search(r"(https?://|www\.|\.[a-z]{2,3}/|\b[a-z0-9-]+\.(com|net|org|ng|co|io|ly|do)\b)", msg.lower()) else 0
        has_email = 1 if re.search(r"[\w\.-]+@[\w\.-]+\.\w+", msg) else 0
        has_phone = 1 if re.search(r"(\+?\d[\d\s-]{7,}\d)", msg) else 0
        meta_rows.append([has_url, has_email, has_phone])

    return hstack([text_vec, np.array(meta_rows, dtype=float)])

# Combine your 60 messages from Cell 16
test_texts = scam_msgs + normal_msgs
test_true_binary = ["scam"] * len(scam_msgs) + ["normal"] * len(normal_msgs)

# Predict 3-class labels (ham, smishing, spam) and map to binary (normal vs scam)
X_custom_test = extract_features(test_texts)
raw_preds = best_model.predict(X_custom_test)
pred_binary = ["normal" if p == "ham" else "scam" for p in raw_preds]

print(f"=== REAL-WORLD TEST SET (60 Custom Messages) ===")
print(f"Test Accuracy: {accuracy_score(test_true_binary, pred_binary):.2%}\n")
print(classification_report(test_true_binary, pred_binary))

# Show any messages the model missed so you can inspect errors
print("--- Missed Messages (Error Analysis) ---")
for msg, actual, raw_p, bin_p in zip(test_texts, test_true_binary, raw_preds, pred_binary):
    if actual != bin_p:
        print(f"[Actual: {actual:6s} | Predicted: {raw_p:8s}] {msg}")

=== REAL-WORLD TEST SET (60 Custom Messages) ===
Test Accuracy: 53.33%

              precision    recall  f1-score   support

      normal       0.52      0.90      0.66        30
        scam       0.62      0.17      0.26        30

    accuracy                           0.53        60
   macro avg       0.57      0.53      0.46        60
weighted avg       0.57      0.53      0.46        60

--- Missed Messages (Error Analysis) ---
[Actual: scam   | Predicted: ham     ] Dear customer, your account will be blocked today. Reply with your BVN and ATM PIN to keep it active.
[Actual: scam   | Predicted: ham     ] Hello, I mistakenly sent money to your account. Please send back the code that was sent to your phone so I can reverse it.
[Actual: scam   | Predicted: ham     ] Congratulations! You have been selected for a N500,000 grant. Send your account number and BVN to receive it.
[Actual: scam   | Predicted: ham     ] Please read me the 6 digit number you just received, I need it to con

In [25]:
import re
import numpy as np
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

def extract_rich_features(messages, fit=False):
    # 1. Text TF-IDF (word + character n-grams catch slight variations & Nigerian terms)
    if fit:
        text_vec = vectorizer_new.fit_transform(messages)
    else:
        text_vec = vectorizer_new.transform(messages)

    # 2. Structural & Behavioral Security Signals (learned patterns, no hardcoded test strings)
    meta_rows = []
    for msg in messages:
        m = msg.lower()
        has_url = 1 if re.search(r"(https?://|www\.|\b[a-z0-9-]+\.(com|net|org|ng|co|io|ly|do)\b)", m) else 0
        has_email = 1 if re.search(r"[\w\.-]+@[\w\.-]+\.\w+", m) else 0
        has_phone = 1 if re.search(r"(\+?\d[\d\s-]{7,}\d)", m) else 0

        # Credential / sensitive info harvesting signals
        asks_secret = 1 if re.search(r"\b(pin|otp|cvv|bvn|nin|password|passcode|login|card details|card number|3 digits|verification code|6 digit|code)\b", m) else 0
        # Action / forwarding verbs ("send me", "reply with", "forward", "read me", "enter your")
        asks_action = 1 if re.search(r"\b(send|reply|forward|read me|give|enter|confirm|verify|click|pay|transfer)\b", m) else 0
        # Urgency / financial lure signals
        urgency_lure = 1 if re.search(r"\b(blocked|suspended|deactivated|expired|urgent|urgently|immediately|grant|giveaway|double|free|winner|claim|fee)\b", m) else 0

        meta_rows.append([has_url, has_email, has_phone, asks_secret, asks_action, urgency_lure])

    return hstack([text_vec, np.array(meta_rows, dtype=float)])

# Re-extract features for Train (75%) and Validation (25%)
X_train_rich = extract_rich_features(X_train_t, fit=True)
X_val_rich = extract_rich_features(X_val_t, fit=False)

clf_rich = LogisticRegression(max_iter=1000, random_state=42)
clf_rich.fit(X_train_rich, y_train)

print("Validation Accuracy (75/25 split):", accuracy_score(y_val, clf_rich.predict(X_val_rich)))

# Evaluate on your untouched 60-message test set using combined scam probability (smishing + spam)
X_test_rich = extract_rich_features(test_texts, fit=False)
probs = clf_rich.predict_proba(X_test_rich)
ham_idx = list(clf_rich.classes_).index("ham")
scam_probs = 1.0 - probs[:, ham_idx]  # Total probability of being smishing OR spam

# Use a security-sensitive threshold (e.g., 0.35 instead of 0.50)
pred_binary_rich = ["scam" if p >= 0.35 else "normal" for p in scam_probs]

print(f"\n=== UPDATED REAL-WORLD TEST SET (60 Custom Messages) ===")
print(f"Test Accuracy: {accuracy_score(test_true_binary, pred_binary_rich):.2%}\n")
print(classification_report(test_true_binary, pred_binary_rich))

print("--- Remaining Missed Messages ---")
for msg, actual, pred, sp in zip(test_texts, test_true_binary, pred_binary_rich, scam_probs):
    if actual != pred:
        print(f"[Actual: {actual:6s} | Pred: {pred:6s} | ScamProb: {sp:.2f}] {msg}")

Validation Accuracy (75/25 split): 0.966248037676609

=== UPDATED REAL-WORLD TEST SET (60 Custom Messages) ===
Test Accuracy: 81.67%

              precision    recall  f1-score   support

      normal       0.77      0.90      0.83        30
        scam       0.88      0.73      0.80        30

    accuracy                           0.82        60
   macro avg       0.83      0.82      0.82        60
weighted avg       0.83      0.82      0.82        60

--- Remaining Missed Messages ---
[Actual: scam   | Pred: normal | ScamProb: 0.20] Hello, I mistakenly sent money to your account. Please send back the code that was sent to your phone so I can reverse it.
[Actual: scam   | Pred: normal | ScamProb: 0.18] Please read me the 6 digit number you just received, I need it to confirm something for you.
[Actual: scam   | Pred: normal | ScamProb: 0.24] Invest N20,000 today and receive N200,000 in 24 hours. Send your bank details now to join.
[Actual: scam   | Pred: normal | ScamProb: 0.20] Br

In [26]:
import joblib

def predict_message(text, threshold=0.35):
    features = extract_rich_features([text], fit=False)
    probs = clf_rich.predict_proba(features)[0]
    classes = list(clf_rich.classes_)

    ham_prob = probs[classes.index("ham")]
    smishing_prob = probs[classes.index("smishing")]
    spam_prob = probs[classes.index("spam")]
    total_scam_prob = 1.0 - ham_prob

    verdict = "SCAM / PHISHING" if total_scam_prob >= threshold else "NORMAL (SAFE)"

    print(f"Verdict: {verdict} (Scam Risk: {total_scam_prob:.1%})")
    print(f"Breakdown -> Ham: {ham_prob:.1%} | Smishing: {smishing_prob:.1%} | Spam: {spam_prob:.1%}")
    print(f"Message: {text}")
    print("-" * 60)

# Quick demo on unseen examples
demo_messages = [
    "GTBank Alert: Your BVN needs re-validation today. Reply with your ATM PIN to avoid restriction.",
    "Bro, are we still meeting at the faculty library by 2pm for the Python assignment?",
    "Your OTP is 918274. Do not share this code with anyone, not even bank staff."
]

for msg in demo_messages:
    predict_message(msg)

# Save model & vectorizer for deployment
joblib.dump({"model": clf_rich, "vectorizer": vectorizer_new, "threshold": 0.35}, "sms_phishing_detector.joblib")
print("Saved model bundle to sms_phishing_detector.joblib")

Verdict: SCAM / PHISHING (Scam Risk: 37.0%)
Breakdown -> Ham: 63.0% | Smishing: 20.8% | Spam: 16.2%
Message: GTBank Alert: Your BVN needs re-validation today. Reply with your ATM PIN to avoid restriction.
------------------------------------------------------------
Verdict: NORMAL (SAFE) (Scam Risk: 7.4%)
Breakdown -> Ham: 92.6% | Smishing: 1.3% | Spam: 6.1%
Message: Bro, are we still meeting at the faculty library by 2pm for the Python assignment?
------------------------------------------------------------
Verdict: NORMAL (SAFE) (Scam Risk: 11.1%)
Breakdown -> Ham: 88.9% | Smishing: 6.2% | Spam: 5.0%
Message: Your OTP is 918274. Do not share this code with anyone, not even bank staff.
------------------------------------------------------------
Saved model bundle to sms_phishing_detector.joblib
